In [ ]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/quadruped')

import numpy as np
import casadi as ca
import osqp
from scipy import sparse
import pinocchio as pin

from core    import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel, LEG_KEYS, LEG_IDX
from control import GaitScheduler, LegController, SRBDModel

URDF = '/home/frlab/mj_opt/models/go2/urdf/go2_description.urdf'
PKG  = '/home/frlab/mj_opt/models/go2'
MJCF = '/home/frlab/mj_opt/models/scenes/go2/scene.xml'

%load_ext autoreload
%autoreload 2

##### SRBD 행렬 준비

In [ ]:
import mujoco

state   = FloatingBaseRobotState()
wrapper = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel  = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({'FL': 'FL', 'FR': 'FR', 'RL': 'RL', 'RR': 'RR'})
srbd    = SRBDModel(wrapper, list(LEG_KEYS))

# keyframe 자세에서 한 번 forward
kernel.reset_to_keyframe()
kernel.update_robot_state(state)
wrapper.update_model(state.q, state.dq)

srbd.update_continuous_dynamics()
dt_mpc = 0.02   # 50Hz MPC
srbd.update_discrete_dynamics(dt=dt_mpc)

A_d = srbd.A_d   # (13, 13)
B_d = srbd.B_d   # (13, 12)

nx, nu = A_d.shape[0], B_d.shape[1]
print(f"nx={nx}, nu={nu}, dt={dt_mpc}")
print("A_d =\n", A_d)
print("B_d =\n", B_d)

##### CasADi MPC

In [ ]:
N = 10   # 예측 horizon

A_ca = ca.DM(A_d)
B_ca = ca.DM(B_d)

x0_sym = ca.SX.sym('x0', nx)

X = ca.SX.sym('X', nx, N+1)
U = ca.SX.sym('U', nu, N)
z = ca.vertcat(ca.reshape(X, -1, 1), ca.reshape(U, -1, 1))

Q_diag = np.ones(nx); Q_diag[3:6] = 10.0
Q_ca   = ca.DM(np.diag(Q_diag))
R_ca   = ca.DM(np.eye(nu) * 1e-4)

# 실제 로봇 상태로 x_ref 구성 — x=0이 아닌 현재 상태 유지가 목표
rpy   = pin.rpy.matrixToRpy(wrapper.oMb.rotation)
p_com = wrapper.p_com_W
omega = wrapper.oMb.rotation.T @ state.dq[3:6]
v_com = wrapper.v_com_W
x_ref = ca.DM(np.concatenate([rpy, p_com, omega, v_com, [1.0]]))

# 목적함수: x - x_ref 기준
cost = 0
for k in range(N):
    dx = X[:, k] - x_ref
    cost += dx.T @ Q_ca @ dx + U[:, k].T @ R_ca @ U[:, k]
cost += (X[:, N] - x_ref).T @ Q_ca @ (X[:, N] - x_ref)

# 등식 제약
g_list = [X[:, 0] - x0_sym]
for k in range(N):
    g_list.append(X[:, k+1] - (A_ca @ X[:, k] + B_ca @ U[:, k]))
g_expr = ca.vertcat(*g_list)

H_sym = ca.hessian(cost, z)[0]
A_sym = ca.jacobian(g_expr, z)
b_sym = -ca.substitute(g_expr, z, ca.DM.zeros(z.shape))

get_qp = ca.Function('get_qp', [x0_sym], [H_sym, A_sym, b_sym])

x0_val = ca.DM(x_ref)   # ca.DM은 .copy() 없음
print("x0 [rpy, p, omega, v, g_aug]:\n", np.array(x0_val).flatten())

H_np, A_np, b_np = get_qp(x0_val)
H_np = np.array(H_np)
A_np = np.array(A_np)
b_np = np.array(b_np).flatten()

print(f"H: {H_np.shape}, A: {A_np.shape}, b: {b_np.shape}")


##### OSQP로 풀기

In [ ]:
# OSQP: min 0.5 z'Hz + g'z  s.t. l <= Az <= u
prob = osqp.OSQP()
prob.setup(
    sparse.csc_matrix(H_np),
    np.zeros(H_np.shape[0]),        # g=0 (목적함수 선형항 없음)
    sparse.csc_matrix(A_np),
    b_np,                           # l = b (등식)
    b_np,                           # u = b (등식)
    warm_starting=True,
    verbose=False,
    eps_abs=1e-6,
    eps_rel=1e-6,
)

res = prob.solve()
print("OSQP status:", res.info.status)

z_sol = res.x
X_sol = z_sol[:nx*(N+1)].reshape(N+1, nx)
U_sol = z_sol[nx*(N+1):].reshape(N, nu)

u_first = U_sol[0]   # 첫 번째 스텝 GRF (12,)
print("첫 번째 GRF (FL, FR, RL, RR x,y,z):\n", u_first.reshape(4, 3))

##### 상태 궤적 시각화

In [ ]:
import matplotlib.pyplot as plt

t_arr = np.arange(N+1) * dt_mpc

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

axes[0].plot(t_arr, X_sol[:, 3], label='p_x')
axes[0].plot(t_arr, X_sol[:, 4], label='p_y')
axes[0].plot(t_arr, X_sol[:, 5], label='p_z')
axes[0].set_ylabel('CoM position [m]')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t_arr, X_sol[:, 0], label='roll')
axes[1].plot(t_arr, X_sol[:, 1], label='pitch')
axes[1].plot(t_arr, X_sol[:, 2], label='yaw')
axes[1].set_ylabel('orientation [rad]')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(t_arr[:-1], U_sol[:, 2],  label='FL Fz')
axes[2].plot(t_arr[:-1], U_sol[:, 5],  label='FR Fz')
axes[2].plot(t_arr[:-1], U_sol[:, 8],  label='RL Fz')
axes[2].plot(t_arr[:-1], U_sol[:, 11], label='RR Fz')
axes[2].set_ylabel('GRF Fz [N]')
axes[2].set_xlabel('Time [s]')
axes[2].legend(); axes[2].grid(True)

plt.suptitle('SRBD MPC (CasADi + OSQP)')
plt.tight_layout()
plt.show()

##### Kinodynamics MPC (Go2 full model, 초기 상태 한 번 선형화)

In [ ]:
import pinocchio as pin

# ── 초기 상태 (keyframe) ───────────────────────────────────────────────
q0  = state.q.copy()    # (19,)  floating base 7 + 관절 12
dq0 = state.dq.copy()  # (18,)

wrapper.update_model(q0, dq0)

# ── Pinocchio로 수치 행렬 계산 ────────────────────────────────────────
M_pin = wrapper.data.M.copy()    # (18, 18)
nle   = wrapper.nle.copy()       # (18,)

# 접촉 발 자코비안 J_c — fid 키는 "FL"/"FR"/"RL"/"RR"
J_c_list = []
for leg in LEG_KEYS:
    J = wrapper.compute_J_W(leg)   # (6, 18)
    J_c_list.append(J[:3])         # 선속도만 (3, 18)
J_c = np.vstack(J_c_list)          # (12, 18)

# 액추에이터 선택 행렬 S: floating base(6) 제외, 관절(12)만
S = np.zeros((12, 18))
S[:, 6:] = np.eye(12)

# ── 연속 dynamics 선형화 ──────────────────────────────────────────────
M_inv = np.linalg.inv(M_pin)
nx_kd = 19 + 18   # 37
nu_kd = 12 + 12   # 24  (tau + lambda)

A_c_kd = np.zeros((nx_kd, nx_kd))
A_c_kd[:19, 19:] = np.eye(19)[:19, :18]

B_c_kd = np.zeros((nx_kd, nu_kd))
B_c_kd[19:, :12] = M_inv @ S.T     # tau
B_c_kd[19:, 12:] = M_inv @ J_c.T  # lambda

# ── Euler 이산화 ──────────────────────────────────────────────────────
dt_kd = 0.02
A_kd  = np.eye(nx_kd) + A_c_kd * dt_kd
B_kd  = B_c_kd * dt_kd

print(f"Kinodynamics: nx={nx_kd}, nu={nu_kd}")
print(f"A_kd: {A_kd.shape}, B_kd: {B_kd.shape}")


In [ ]:
N_kd = 10

A_ca_kd = ca.DM(A_kd)
B_ca_kd = ca.DM(B_kd)

x0_kd = ca.SX.sym('x0_kd', nx_kd)
X_kd  = ca.SX.sym('X_kd', nx_kd, N_kd+1)
U_kd  = ca.SX.sym('U_kd', nu_kd, N_kd)
z_kd  = ca.vertcat(ca.reshape(X_kd, -1, 1), ca.reshape(U_kd, -1, 1))

# 가중치: quaternion 제외, rpy 대신 위치/관절각만
Q_kd_diag = np.zeros(nx_kd)
Q_kd_diag[0:3]   = 100.0   # base position (x,y,z)
Q_kd_diag[7:19]  = 1.0     # 관절각
Q_kd_diag[19:22] = 1.0     # base linear vel
Q_kd_diag[25:37] = 0.1     # 관절 속도
Q_ca_kd = ca.DM(np.diag(Q_kd_diag))

R_kd_diag = np.zeros(nu_kd)
R_kd_diag[:12]  = 1e-3     # tau
R_kd_diag[12:]  = 1e-4     # lambda
R_ca_kd = ca.DM(np.diag(R_kd_diag))

x_eq_kd = ca.DM(np.concatenate([q0, dq0]))

cost_kd = 0
for k in range(N_kd):
    dx = X_kd[:, k] - x_eq_kd
    cost_kd += dx.T @ Q_ca_kd @ dx + U_kd[:, k].T @ R_ca_kd @ U_kd[:, k]
cost_kd += (X_kd[:, N_kd] - x_eq_kd).T @ Q_ca_kd @ (X_kd[:, N_kd] - x_eq_kd)

# 등식 제약: dynamics
g_kd = [X_kd[:, 0] - x0_kd]
for k in range(N_kd):
    g_kd.append(X_kd[:, k+1] - (A_ca_kd @ X_kd[:, k] + B_ca_kd @ U_kd[:, k]))
g_kd_expr = ca.vertcat(*g_kd)

# QP 행렬 추출
H_kd_sym = ca.hessian(cost_kd, z_kd)[0]
A_eq_sym  = ca.jacobian(g_kd_expr, z_kd)
b_eq_sym  = -ca.substitute(g_kd_expr, z_kd, ca.DM.zeros(z_kd.shape))

get_qp_kd = ca.Function('get_qp_kd', [x0_kd], [H_kd_sym, A_eq_sym, b_eq_sym])

x0_kd_val = ca.DM(np.concatenate([q0, dq0]))
H_kd_np, A_eq_np, b_eq_np = get_qp_kd(x0_kd_val)
H_kd_np  = np.array(H_kd_np)
A_eq_np  = np.array(A_eq_np)
b_eq_np  = np.array(b_eq_np).flatten()

# 부등식 제약 추가: λ_z >= 0 (4발), |τ| <= τ_max
# OSQP: l <= [A_eq; A_ineq] z <= u
tau_max = 23.7   # Go2 관절 토크 한계 [Nm]
n_z     = H_kd_np.shape[0]
nU      = N_kd * nu_kd

# τ 인덱스: z = [X(37×11), U(24×10)] → U 시작 인덱스
u_start = nx_kd * (N_kd + 1)

# τ 제약: -tau_max <= τ_k <= tau_max  (N_kd×12 개)
A_tau = np.zeros((N_kd * 12, n_z))
for k in range(N_kd):
    row0 = k * 12
    col0 = u_start + k * nu_kd
    A_tau[row0:row0+12, col0:col0+12] = np.eye(12)

# λ_z >= 0 제약: 각 스텝 4발 z 성분 (인덱스 14, 17, 20, 23)
lam_z_idx = [14, 17, 20, 23]   # nu_kd 내 lambda_z 위치
A_lam = np.zeros((N_kd * 4, n_z))
for k in range(N_kd):
    for j, idx in enumerate(lam_z_idx):
        row = k * 4 + j
        col = u_start + k * nu_kd + idx
        A_lam[row, col] = 1.0

A_ineq = np.vstack([A_tau, A_lam])
l_tau  = np.full(N_kd * 12, -tau_max)
u_tau  = np.full(N_kd * 12,  tau_max)
l_lam  = np.zeros(N_kd * 4)
u_lam  = np.full(N_kd * 4, np.inf)

A_full = np.vstack([A_eq_np, A_ineq])
l_full = np.concatenate([b_eq_np, l_tau, l_lam])
u_full = np.concatenate([b_eq_np, u_tau, u_lam])

prob_kd = osqp.OSQP()
prob_kd.setup(
    sparse.csc_matrix(H_kd_np),
    np.zeros(H_kd_np.shape[0]),
    sparse.csc_matrix(A_full),
    l_full, u_full,
    warm_starting=True,
    verbose=False,
    eps_abs=1e-6, eps_rel=1e-6,
)

res_kd = prob_kd.solve()
print("OSQP status:", res_kd.info.status)

z_kd_sol     = res_kd.x
X_kd_sol     = z_kd_sol[:nx_kd*(N_kd+1)].reshape(N_kd+1, nx_kd)
U_kd_sol     = z_kd_sol[nx_kd*(N_kd+1):].reshape(N_kd, nu_kd)
tau_first    = U_kd_sol[0, :12]
lambda_first = U_kd_sol[0, 12:]
print("첫 스텝 τ  (12관절) [Nm]:\n", np.round(tau_first, 3))
print("첫 스텝 λ  (4발 xyz) [N]:\n", np.round(lambda_first.reshape(4, 3), 3))
print(f"총 Fz: {lambda_first.reshape(4,3)[:,2].sum():.1f} N  (중력: {wrapper.m*9.81:.1f} N)")
